# 02 Data Cleaning


The goal is to turn the raw Bank Marketing data into a clean, analysis-ready interim dataset while keeping the raw data unchanged.

## Cleaning Goals

From the exploration notebook, we know that the main cleaning tasks are:

- Load the raw dataset.
- Preserve the raw data unchanged.
- Validate data types.
- Encode the target variable (y) as a binary feature.
- Check duplicate rows.
- Inspect special values (unknown and pdays = -1).
- Save the cleaned dataset to data/interim/.

We keep the cleaning logic in this notebook for now. Refactoring into `src/` comes later as a separate learning step.

## Cleaning Decisions

These decisions are deliberately simple and visible:

| Issue | Decision | Reason |
| --- | --- | --- |
| `Target (y)` is `Yes` or `No` | Encode as 0/1 | Required for modelling |
| Duplicate rows | Validate | Ensure dataset quality |
| unknown values |Inspect and preserve for now | Legitimate category, not missing values |
| pdays = -1 | Document as special code | Means no previous campaign |
| Data types | Validate | Confirm correct import |

## 1. Setup

In [22]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)


def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "data" / "raw" / "bank-full.csv").exists():
            return path
    raise FileNotFoundError("Could not find project root with data/raw/bank-full.csv")


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "bank-full.csv"
INTERIM_DATA_PATH = PROJECT_ROOT / "data" / "interim" / "bank_marketing_cleaned.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data path: {RAW_DATA_PATH}")
print(f"Interim data path: {INTERIM_DATA_PATH}")

Project root: /home/patri/master/ads2-bank-marketing-project
Raw data path: /home/patri/master/ads2-bank-marketing-project/data/raw/bank-full.csv
Interim data path: /home/patri/master/ads2-bank-marketing-project/data/interim/bank_marketing_cleaned.csv


## 2. Load Raw Data

In [23]:
def load_raw_data(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Raw data file not found: {path}")

    df = pd.read_csv(path, sep = ";")
    print(f"Loaded raw data with {df.shape[0]:,} rows and {df.shape[1]:,} columns.")
    return df


raw_df = load_raw_data(RAW_DATA_PATH)
raw_df.head()

Loaded raw data with 45,211 rows and 17 columns.


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


## 3. Inspect Known Quality Issues

#### Data Validation Before Cleaning

Before modifying the dataset, we validate the main assumptions identified during the exploration phase.

The checks confirm that:

- No duplicate rows are present.
- The target variable contains only the expected values ("yes" and "no").
- No standard missing values are present in the dataset.

In [24]:
print(f"Duplicate rows: {raw_df.duplicated().sum()}")
print(f"Target values: {sorted(raw_df['y'].dropna().unique())}")
print(f"Missing values: {raw_df.isna().sum().sum()}")

Duplicate rows: 0
Target values: ['no', 'yes']
Missing values: 0


#### Special Values

In [25]:
unknown_counts = {
    column: (raw_df[column] == "unknown").sum()
    for column in raw_df.select_dtypes(include="object").columns
    if (raw_df[column] == "unknown").sum() > 0
}

unknown_counts

{'job': np.int64(288),
 'education': np.int64(1857),
 'contact': np.int64(13020),
 'poutcome': np.int64(36959)}

The dataset contains the category "unknown" in several categorical variables.  
These are valid business values rather than missing values, so they will be preserved during cleaning.

In [30]:
pdays_minus_one_count = (raw_df["pdays"] == -1).sum()

print(f"Rows with pdays = -1: {pdays_minus_one_count:,}")

Rows with pdays = -1: 36,954


The value -1 in the `pdays` variable is a documented special code indicating that the client had not been contacted in a previous marketing campaign. Therefore, it will not be treated as a missing value.

#### Key Observations

- No duplicate rows or standard missing values were detected.
- The target variable contains only the expected values: `"no"` and `"yes"`.
- The category `"unknown"` is present in several variables and will be preserved as a valid category.
- The value `pdays = -1` indicates that the client was not contacted in a previous campaign and will also be preserved.
- The feature `duration` will be removed because it is only available after the call and would introduce data leakage.

## 4. Define Cleaning Function

In [31]:
def clean_bank_marketing_data(df: pd.DataFrame) -> pd.DataFrame:
    """Clean the raw Bank Marketing dataset."""

    # Work on a copy so that the raw DataFrame remains unchanged.
    cleaned = df.copy()

    # Remove accidental spaces from column names.
    cleaned.columns = cleaned.columns.str.strip()

    # Validate the columns required for cleaning.
    required_columns = {"y", "duration"}
    missing_columns = required_columns - set(cleaned.columns)

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {sorted(missing_columns)}"
        )

    # Validate duplicate rows.
    if cleaned.duplicated().any():
        raise ValueError(
            "Duplicate rows found. Investigate before continuing."
        )

    # Validate the target values before encoding.
    target_mapping = {"no": 0, "yes": 1}
    unexpected_target_values = (
        set(cleaned["y"].dropna().unique()) - set(target_mapping)
    )

    if unexpected_target_values:
        raise ValueError(
            f"Unexpected target values: {sorted(unexpected_target_values)}"
        )

    # Encode the target while keeping the original y column.
    cleaned["y_binary"] = cleaned["y"].map(target_mapping).astype("int64")

    # Remove the leakage feature.
    cleaned = cleaned.drop(columns=["duration"])

    return cleaned

In [32]:
cleaned_df = clean_bank_marketing_data(raw_df)

cleaned_df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,pdays,previous,poutcome,y,y_binary
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,1,-1,0,unknown,no,0
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,1,-1,0,unknown,no,0
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,1,-1,0,unknown,no,0
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,1,-1,0,unknown,no,0
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,1,-1,0,unknown,no,0


#### Key Observations

- The cleaning function preserves the raw dataset by working on a copy.
- The target variable is encoded as `y_binary`, where `"no" = 0` and `"yes" = 1`.
- The original target column `y` is preserved for traceability.
- The feature `duration` is removed because it would introduce data leakage.
- Special values such as `"unknown"` and `pdays = -1` remain unchanged because they have documented business meaning.

## 5. Validate Cleaned Data

In [33]:
print(f"Raw shape: {raw_df.shape}")
print(f"Cleaned shape: {cleaned_df.shape}")
print(f"Duration present: {'duration' in cleaned_df.columns}")
print(f"Binary target values: {sorted(cleaned_df['y_binary'].unique())}")

Raw shape: (45211, 17)
Cleaned shape: (45211, 17)
Duration present: False
Binary target values: [np.int64(0), np.int64(1)]


In [34]:
validation_summary = pd.DataFrame({
    "missing_count": cleaned_df.isna().sum(),
    "dtype": cleaned_df.dtypes.astype(str),
    "unique_values": cleaned_df.nunique(),
})

validation_summary

,missing_count,dtype,unique_values
age,0,int64,77
job,0,object,12
marital,0,object,3
education,0,object,4
default,0,object,2
balance,0,int64,7168
housing,0,object,2
loan,0,object,2
contact,0,object,3
day,0,int64,31


In [35]:
pd.DataFrame({
    "label": ["no", "yes"],
    "encoded_value": [0, 1],
    "count": [
        (cleaned_df["y"] == "no").sum(),
        (cleaned_df["y"] == "yes").sum(),
    ],
})

,label,encoded_value,count
0,no,0,39922
1,yes,1,5289


#### Key Observations

- The cleaned dataset contains no missing values.
- The leakage feature `duration` has been removed.
- The binary target (`y_binary`) has been successfully encoded while preserving the original target column.
- The number of records remains unchanged after cleaning.

## 6. Save Interim Dataset

In [36]:
INTERIM_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
cleaned_df.to_csv(INTERIM_DATA_PATH, index=False)

print(f"Saved cleaned interim data to: {INTERIM_DATA_PATH}")

Saved cleaned interim data to: /home/patri/master/ads2-bank-marketing-project/data/interim/bank_marketing_cleaned.csv


In [37]:
reloaded_df = pd.read_csv(INTERIM_DATA_PATH)

print(f"Reloaded shape: {reloaded_df.shape}")
reloaded_df.head()

Reloaded shape: (45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,pdays,previous,poutcome,y,y_binary
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,1,-1,0,unknown,no,0
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,1,-1,0,unknown,no,0
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,1,-1,0,unknown,no,0
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,1,-1,0,unknown,no,0
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,1,-1,0,unknown,no,0


## Cleaning Summary

The cleaned dataset is now ready for feature engineering.

Key changes:

- Encoded the target variable (`y`) into `y_binary`.
- Preserved the original target column.
- Removed the leakage feature `duration`.
- Preserved the business values `"unknown"` and `pdays = -1`.
- No duplicate rows or missing values required cleaning.